# Проверка обновлённого вида КЕДР

Вид `sbx_edm_dfip.v_detail_dmkb_dkb_dmsb` после правки коллег должен отдавать **все** продукты клиентов, у которых есть торговый эквайринг.

1. Определение вида: какой теперь фильтр.
2. Какие продукты есть в виде (должно быть много, а не только «Торговый эквайринг»).
3. Контрольный клиент 1207007950 за 202607: ожидаем 64 311 307 (как в озере `Kedr.v_detail_dkb`).
4. Месяцы в виде: строки, клиенты, сумма.
5. Сверка со старой таблицей `sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month` за 202601–202607.
6. Права технической учётки DAG на вид.

Только чтение. Логин и пароль DRP и логин keytab спрашиваются один раз. На скриншотах логины закрывайте `***`.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:,.0f}".format)

DRP_VIEW = "sbx_edm_dfip.v_detail_dmkb_dkb_dmsb"
OLD_TABLE = "sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month"
MONTH = 202607
CHECK_INN = "1207007950"
CHECK_INN_EXPECTED = 64_311_307
CMP_FROM, CMP_TO = 202601, 202607

if "drp" not in globals():
    drp = connect(
        to="DRP",
        user_params={"user_name": input("DRP login: ").strip(), "password": getpass("DRP password: ")},
    )

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def drp_fetch(sql):
    with drp:
        return drp.fetch(sql)


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=4g")
        return imp.fetch(sql)


def norm_inn(v):
    s = "".join(ch for ch in str(v) if ch.isdigit())
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s or None

## 1. Определение вида

Раньше было `AND z.prod_name = 'Торговый эквайринг'` (фильтр строк). Теперь должен быть фильтр клиентов, например `inn IN (SELECT ... WHERE prod_name = 'Торговый эквайринг')`.

In [ ]:
viewdef = drp_fetch(f"SELECT pg_get_viewdef('{DRP_VIEW}'::regclass, true) AS def")
print(viewdef["def"].iloc[0])

## 2. Какие продукты есть в виде за месяц

Должно быть много строк: кредиты, депозиты, РКО и т. д. Если осталась одна строка «Торговый эквайринг», правка не применилась.

In [ ]:
products = drp_fetch(f"""
    SELECT prod_name, product, count(*) AS rows_cnt, count(DISTINCT inn) AS inn_cnt, sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm = {MONTH}
    GROUP BY prod_name, product
    ORDER BY nbi DESC
""")
print("Разных prod_name:", products["prod_name"].nunique(), "| итого nbi_ssp:", f"{products['nbi'].sum():,.0f}")
display(products.head(40))

## 3. Контрольный клиент

В озере (`Kedr.v_detail_dkb`) у ИНН 1207007950 за 202607 было 64 311 307, из них «Торговый эквайринг» 20 800.

In [ ]:
one = drp_fetch(f"""
    SELECT zo_itog, prod_name, product, count(*) AS rows_cnt, sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm = {MONTH} AND inn = '{CHECK_INN}'
    GROUP BY zo_itog, prod_name, product
    ORDER BY nbi DESC
""")
display(one)
total = float(pd.to_numeric(one["nbi"], errors="coerce").sum()) if len(one) else 0.0
print(f"Итого: {total:,.0f} | ожидали: {CHECK_INN_EXPECTED:,.0f} | разница: {total - CHECK_INN_EXPECTED:,.0f}")
print("OK" if abs(total - CHECK_INN_EXPECTED) < 1 else "НЕ СОВПАДАЕТ")

## 4. Месяцы в виде

`inn_with_acq` — сколько клиентов в этом месяце имеют строку «Торговый эквайринг». Если `inn_cnt` больше, в вид попали и клиенты без дохода по эквайрингу в этом месяце (так и задумано, если коллеги отбирали клиентов по всем месяцам).

In [ ]:
months = drp_fetch(f"""
    SELECT yearmm,
           count(*) AS rows_cnt,
           count(DISTINCT inn) AS inn_cnt,
           count(DISTINCT CASE WHEN prod_name = 'Торговый эквайринг' THEN inn END) AS inn_with_acq,
           count(DISTINCT prod_name) AS prod_names,
           sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    GROUP BY yearmm
    ORDER BY yearmm
""")
display(months)

## 5. Сверка со старой таблицей за 202601–202607

Старая таблица собрана из трёх видов озера (`Kedr.v_detail_dmkb/dmsb/dkb`) и берёт **максимум** по видам, а не сумму. Поэтому из вида DRP считаем два варианта на клиента и месяц:
- `view_sum` — сумма по всем ЗО;
- `view_max_zo` — максимум из сумм по каждому ЗО (как в старой таблице).

Совпадение считаем с точностью до 1 рубля.

In [ ]:
view_zo = drp_fetch(f"""
    SELECT yearmm, inn, zo_itog, sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm BETWEEN {CMP_FROM} AND {CMP_TO}
    GROUP BY yearmm, inn, zo_itog
""")
view_zo["inn"] = view_zo["inn"].map(norm_inn)
view_zo["yearmm"] = view_zo["yearmm"].astype(int)
view_zo["nbi"] = pd.to_numeric(view_zo["nbi"], errors="coerce").fillna(0)
view_inn = (
    view_zo.groupby(["yearmm", "inn"], as_index=False)
    .agg(view_sum=("nbi", "sum"), view_max_zo=("nbi", "max"), zo_cnt=("zo_itog", "nunique"))
)

old = imp_fetch(f"""
    SELECT cast(yearmm AS bigint) AS yearmm, cast(inn AS string) AS inn,
           max(cast(kedr_obshiy_chod AS double)) AS old_value
    FROM {OLD_TABLE}
    WHERE cast(yearmm AS bigint) BETWEEN {CMP_FROM} AND {CMP_TO}
      AND kedr_obshiy_chod IS NOT NULL
    GROUP BY cast(yearmm AS bigint), cast(inn AS string)
""")
old["inn"] = old["inn"].map(norm_inn)
old["yearmm"] = old["yearmm"].astype(int)
old = old.groupby(["yearmm", "inn"], as_index=False)["old_value"].max()

cmp = view_inn.merge(old, on=["yearmm", "inn"], how="outer", indicator=True)
both = cmp[cmp["_merge"] == "both"].copy()
both["match_sum"] = (both["view_sum"] - both["old_value"]).abs() < 1
both["match_max_zo"] = (both["view_max_zo"] - both["old_value"]).abs() < 1

summary = (
    cmp.groupby("yearmm")
    .apply(lambda g: pd.Series({
        "inn_view": int((g["_merge"] != "right_only").sum()),
        "inn_old": int((g["_merge"] != "left_only").sum()),
        "inn_both": int((g["_merge"] == "both").sum()),
        "only_view": int((g["_merge"] == "left_only").sum()),
        "only_old": int((g["_merge"] == "right_only").sum()),
    }))
    .join(
        both.groupby("yearmm").agg(
            match_sum_pct=("match_sum", lambda s: round(100 * s.mean(), 2)),
            match_max_zo_pct=("match_max_zo", lambda s: round(100 * s.mean(), 2)),
            view_sum_total=("view_sum", "sum"),
            view_max_zo_total=("view_max_zo", "sum"),
            old_total=("old_value", "sum"),
        )
    )
    .reset_index()
)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Сводка по месяцам (суммы — только по общим ИНН):")
display(summary)

print("Топ-20 расхождений (по модулю разницы с view_max_zo):")
both["diff"] = both["view_max_zo"] - both["old_value"]
display(
    both.loc[~both["match_max_zo"]]
    .assign(abs_diff=lambda d: d["diff"].abs())
    .sort_values("abs_diff", ascending=False)
    [["yearmm", "inn", "zo_cnt", "view_sum", "view_max_zo", "old_value", "diff"]]
    .head(20)
)
pd.set_option("display.float_format", "{:,.0f}".format)

## 6. Права технической учётки DAG

Введите логин из Airflow Connection `spark_drp` (не личный). Пусто — пропустить.

In [ ]:
tech_user = input("Логин технической учётки (spark_drp), Enter — пропустить: ").strip()
if tech_user:
    schema = DRP_VIEW.split(".")[0]
    display(drp_fetch(f"""
        SELECT has_schema_privilege('{tech_user}', '{schema}', 'USAGE') AS usage_ok,
               has_table_privilege('{tech_user}', '{DRP_VIEW}', 'SELECT') AS select_ok
    """))
else:
    print("Пропущено")